# 03 — Agreement between UQ methods

Three views of whether the methods agree on *which* samples are uncertain: overlap
of the 100 most uncertain samples per class (Fig. 6), Kendall's tau on the score
rankings (Fig. 7) and a KS test on the score distributions (Fig. 8). All three are
pooled over the five seeds.

In [ ]:
import os, sys, pathlib

# repo root, whether the notebook is run from notebooks/ or from the root
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

# released data: the summary bundle (all notebooks) and the raw per-pass predictions
# (notebooks 04 and 06). Set the variables before starting Jupyter to use other locations.
os.environ.setdefault('UQFIG_BUNDLE', str(ROOT / 'figure_data_bundle'))
os.environ.setdefault('UQFIG_PREDS', str(ROOT / 'cluster_preds'))
OUT = ROOT / 'out'; OUT.mkdir(exist_ok=True)

from uq_genomics.figures import style as S
S.apply_rc()
print('bundle :', os.environ['UQFIG_BUNDLE'])
print('preds  :', os.environ['UQFIG_PREDS'])
print('output :', OUT)

In [ ]:
from uq_genomics.figures import combine_figures as CF
CF.top100(str(OUT))      # pooled over seeds: bars = mean, error bars = SD
CF.kendall(str(OUT))
CF.ks(str(OUT))

In [ ]:
from IPython.display import IFrame, display
p = OUT / 'COMBINED_top100_all_usecases.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=560)) if p.exists() else None
p = OUT / 'COMBINED_kendall_all_usecases.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=560)) if p.exists() else None
p = OUT / 'COMBINED_ks_all_usecases.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=560)) if p.exists() else None

## Zero-variance samples

MC-dropout's score is the SD of the predicted probability over 100 stochastic passes.
With every dropout layer active at inference (the configuration used in the paper;
see Section S2.6 and `scripts/mcd_inference/mcd_inference.py`), fewer than 1% of
the test samples have an SD of exactly zero in any setting. The cell below reports the
fraction per setting as a check on the bundle.

In [ ]:
import numpy as np
from uq_genomics.figures import config_paths as CP, data_access as DA, plots_sensitivity as PS
for task in ['rbp', 'gex', 'tf']:
    comp = DA.load_comparison(task); spec = CP.VERSIONS[task]
    for setting in spec['settings']:
        fr = []
        for seed in spec['seeds']:
            sd, _ = PS._sd_for(comp, task, seed, setting, 'mcd')
            if sd is not None:
                fr.append((np.asarray(sd) == 0).mean())
        if fr:
            print(f'{task:4s} {setting:16s} zero-SD fraction (MCD) = {np.mean(fr):.3f}  over {len(fr)} seeds')